In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import scanpy as sc
from tqdm.auto import tqdm
from wppkg import write_json
from perthub.tokenizer import tokenize_adata_to_hf_dataset

### Step 1: preprocess your adata

- adata.X: normalize_total and log1p

- check and unify smiles; filter low frequency cells

- calculate rdkit2d embeddings; dose and time normalization

- select top-k hvgs if needed

- split train, valid, and test datasets

- add adata.obs["sample_indices"]

In [3]:
adata = sc.read_h5ad("../../data/sciplex3_biolord.h5ad")

# TODO: feel free to add preprocessing if it's required.
...

# add adata.obs["sample_indices"]
adata.obs["sample_indices"] = np.arange(len(adata))

### Step 2: construct `attributes_map`

- `ordered_attributes_map`

- `categorical_attributes_map`

- `n_samples` 

In [ ]:
from collections import defaultdict

attributes_map = defaultdict(dict)

# add `categorical_attributes_map`
categorical_attributes_a2d = {"cell_type": "cell_type"}  # anndata: dataset
for attr_a, attr_d in tqdm(categorical_attributes_a2d.items(), desc="building categorical_attributes_map"):
    attributes_map["categorical_attributes_map"][attr_d] = {cat: i for i, cat in enumerate(adata.obs[attr_a].unique())}

# add `ordered_attributes_map`
ordered_attributes_a2d = {"rdkit2d_dose": "rdkit2d_dose"}  # anndata: dataset
for attr_a, attr_d in tqdm(ordered_attributes_a2d.items(), desc="building ordered_attributes_map"):
    attributes_map["ordered_attributes_map"][attr_d] = adata.obsm[attr_a].shape[-1]

# add `n_samples`
unknown_attributes_a2d = {"sample_indices": "sample_indices"}
# Actually, n_samples just needs to count the total cells from the training and validation sets.
attributes_map["n_samples"] = len(adata)  # just save `n_samples`

# save
write_json(attributes_map, "../../data/attributes_map.json")

building categorical_attributes_map:   0%|          | 0/1 [00:00<?, ?it/s]

building ordered_attributes_map:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
attributes_map

defaultdict(dict,
            {'categorical_attributes_map': {'cell_type': {'A549': 0,
               'MCF7': 1,
               'K562': 2}},
             'ordered_attributes_map': {'rdkit2d_dose': 174},
             'n_samples': 354640})

### Step 3: tokenization

In [6]:
# categorical attributes mapping: str -> int
for attr_a, attr_b in tqdm(categorical_attributes_a2d.items(), desc="converting categorical attributes"):
    adata.obs[attr_a] = adata.obs[attr_a].map(attributes_map["categorical_attributes_map"][attr_b])

converting categorical attributes:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# extract train, valid, test
adata_train = adata[adata.obs["split_ood"] == "train"]
adata_valid = adata[adata.obs["split_ood"] == "test"]
# adata_test = adata[adata.obs["split_ood"] == "ood"]

# tokenize
attr_map_dict = {**categorical_attributes_a2d, **ordered_attributes_a2d, **unknown_attributes_a2d}

ds_train = tokenize_adata_to_hf_dataset(
    adata=adata_train,
    attr_map_dict=attr_map_dict,
    x_key="x"
)
ds_train.save_to_disk("../../data/ds_train")

ds_valid = tokenize_adata_to_hf_dataset(
    adata=adata_valid,
    attr_map_dict=attr_map_dict,
    x_key="x"
)
ds_valid.save_to_disk("../../data/ds_valid")

Tokenizing to HF Dataset:   0%|          | 0/16 [00:00<?, ?it/s]

Saving the dataset (0/6 shards):   0%|          | 0/313598 [00:00<?, ? examples/s]

Tokenizing to HF Dataset:   0%|          | 0/2 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/29192 [00:00<?, ? examples/s]